Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Matching country names: `country_converter`

A world choropleth needs the **same code** for each country in the map and in the data. But country names come in many forms:

| One source says… | Another says… |
|---|---|
| `Korea, Rep.` | `South Korea`, `Republic of Korea` |
| `Cote d'Ivoire` | `Côte d'Ivoire`, `Ivory Coast` |
| `Turkiye` | `Turkey` |

Matching these by hand is slow and error-prone. The [`country_converter`](https://github.com/IndEcol/country_converter) package (`coco`) recognises hundreds of spellings and converts them to standard codes, such as **ISO3** (`GBR`, `KOR`, `CIV`), which map files use.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet
%pip install country_converter --quiet

In [ ]:
import pandas as pd
import altair as alt
import country_converter as coco

<br><br>

---

<br>

## 1. Convert a few names

`coco.convert()` takes a list of names (or codes) and what to convert them `to`:

In [ ]:
names = ["Korea, Rep.", "South Korea", "Ivory Coast", "Cote d'Ivoire", "Turkiye", "UK", "Viet Nam"]

coco.convert(names=names, to="ISO3")

<br>

Different spellings of the same country get the same code.

### What isn't a country?

Data from the World Bank, IMF or OECD often includes **groups** of countries: `World`, `Euro area`, `High income`. These have no shape on a map. `country_converter` says `not found` (and prints a warning), so they're easy to spot and remove:

In [ ]:
coco.convert(names=["France", "World", "Euro area", "Sub-Saharan Africa", "High income"], to="ISO3")

<br><br>

---

<br>

## 2. Other conversions

The same function converts to other classifications too. A few useful ones (the [full list](https://github.com/IndEcol/country_converter#classification-schemes)):

In [ ]:
codes = ["GBR", "FRA", "NGA", "IND", "BRA"]

print(coco.convert(names=codes, to="name_short"))   # short English names, for labels and tooltips
print(coco.convert(names=codes, to="ISO2"))         # two-letter codes
print(coco.convert(names=codes, to="continent"))    # to colour or group by continent

<br><br>

---

<br>

## 3. A whole table

Our World Bank GDP data has country names. Add an ISO3 column with `coco.convert`, then check it against the codes the World Bank gave us:

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week07/data/world_gdp_per_capita_2022.csv")

df["iso3_coco"] = coco.convert(names=df["country"], to="ISO3")
df["continent"] = coco.convert(names=df["iso3_coco"], to="continent")

print(f"Rows: {len(df)}")
print(f"Codes that don't match the World Bank's: {(df['iso3_coco'] != df['iso3']).sum()}")
df.head()

<br>

Any `not found` rows would be groups of countries, or names `coco` doesn't know. Remove the first kind, and fix the second by hand:

In [ ]:
df = df[df["iso3_coco"] != "not found"]

<br><br>

---

<br>

## 4. Map it

Now the data and the map share ISO3 codes. Our world map file has them in `properties.iso_a3_eh` (as in example DS7_15):

In [ ]:
world = alt.topo_feature("https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/world/world.json", "geog")

chart = alt.Chart(world).transform_lookup(
    lookup="properties.iso_a3_eh",
    from_=alt.LookupData(df, key="iso3_coco", fields=["country", "value", "continent"]),
).mark_geoshape(stroke="white", strokeWidth=0.3, invalid="show").encode(
    color=alt.when("!isValid(datum.value)").then(alt.value("#e3e3e3")).otherwise(
        alt.Color("value:Q").title("$").scale(type="log", scheme="viridis")),
    tooltip=[alt.Tooltip("properties.name:N").title("Country"),
             alt.Tooltip("continent:N").title("Continent"),
             alt.Tooltip("value:Q").format("$,.0f").title("GDP per person")],
).project("equalEarth").properties(width=600, height=320, title="GDP per person, 2022 (PPP)")

chart

<br>

> **Codes can still differ between sources.** For example, the World Bank uses `XKX` for Kosovo, while our map file has no ISO3 code for it. After any lookup, check for grey countries that should have data.

> `invalid="show"` keeps countries with no value on the map, so the grey colour can show. Without it, they're left out (a hole).

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Colour the map by `continent` instead of GDP. (Hint: `"continent:N"`.)

**2.** Download a World Bank indicator with the API (Week 4, DS4_2) for all countries. Convert its names with `coco`, remove the groups, and map it.

**3.** `coco` can also tell you which countries are in a group: try `coco.CountryConverter().EU27as("ISO3")` and map just the EU.

In [ ]:
# Your code here
